<a href="https://colab.research.google.com/github/mfvalle/lightning-occurrence/blob/main/Proyecto_Final_Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción Espacio-Temporal de Descargas Eléctricas Atmosféricas en Colombia mediante Redes Neuronales Convolucionales-Recurrentes (ConvLSTM)

**Autores:** Manuel Fernando Valle Amortegui  
**Asignatura:** Redes Neuronales: Arquitecturas y Aplicaciones  
**Profesor:** Arles Rodríguez  
**Fecha de Entrega:** 19 de mayo de 2026  

---

## 1. Introducción
Las descargas eléctricas atmosféricas son fenómenos caracterizados por la transferencia masiva de carga eléctrica, originados principalmente en nubes de tipo *Cumulonimbus* [1], [2]. Aunque la mayor parte de la actividad ocurre en la troposfera, aproximadamente el 25% de estas descargas impactan la superficie terrestre como rayos nube-tierra (CG) [3]. Debido a su ubicación en la Zona de Convergencia Intertropical (ZCIT), su topografía andina y dinámicas locales de vientos, el norte de Sudamérica es catalogado de forma consistente como una de las regiones con mayor actividad de rayos en todo el planeta [4], [5].

En Colombia, este fenómeno constituye una problemática crítica de seguridad pública e infraestructura. Estudios históricos revelaron una tasa de mortalidad anual por rayos de entre 1.51 y 1.78 muertes por millón de habitantes (aproximadamente 80 fatalidades al año), lo que posiciona al país entre las naciones con mayor índice de mortandad por esta causa en Latinoamérica, concentrándose la mayoría de eventos en áreas rurales [6], [7].

El diseño de estrategias de mitigación efectivas requiere determinar con precisión espacio-temporal cuándo y dónde ocurren las descargas [8]. Sin embargo, los sistemas meteorológicos tradicionales fundamentados en Modelos Numéricos de Predicción del Tiempo (NWPM) requieren un altísimo costo computacional y carecen de la agilidad necesaria para alertas a muy corto plazo. Este proyecto propone abordar dicha limitación mediante el desarrollo de un modelo de **Nowcasting** (predicción de 10 a 60 minutos) basado en redes neuronales profundas. Al estructurar los históricos de rayos como mapas de densidad matriciales en el tiempo, se busca entrenar un algoritmo capaz de aprender y predecir la evolución y desplazamiento de las tormentas sobre el territorio colombiano de forma automatizada.

---

## 2. Trabajo Relacionado
El procesamiento y la explotación analítica de datos de descargas eléctricas a gran escala representa un reto computacional mayor. A nivel global, ocurren entre 40 y 50 rayos por segundo, lo que se traduce en millones de impactos diarios que deben ser gestionados [9]. Para contextualizar la magnitud del problema en el ámbito local, los conjuntos de datos de la región central de Colombia superan los 80 millones de registros en ventanas de 5 años, promediando cerca de 40,000 descargas diarias.

Como antecedente directo e inspiración para esta investigación, el desarrollo del aplicativo **Lightspot** en el año 2025 abordó el desafío de la gestión de estos macrodatos utilizando registros de la Red Mundial de Localización de Rayos (**WWLLN**, por sus siglas en inglés) [10]. Lightspot demostró la viabilidad de filtrar, almacenar y mapear eficientemente volúmenes masivos de datos georreferenciados para identificar patrones visuales de tormentas. No obstante, las aproximaciones previas en la literatura frecuentemente se limitan al análisis descriptivo y de visualización histórica de los datos espaciales.

Nuestro proyecto toma como punto de partida el ecosistema de datos provisto por redes de sensores VLF como la WWLLN y la necesidad de explotación identificada por Lightspot, pero **extiende la frontera del estado del arte desde la visualización estática/histórica hacia el modelado predictivo**. Para ello, nos basamos en la arquitectura **ConvLSTM** propuesta originalmente por Shi et al. para *nowcasting* de variables atmosféricas, integrando por primera vez operadores convolucionales sobre las series temporales de la grilla de rayos mapeada en Colombia.

---

## 3. Dataset y Características
El proyecto utiliza datos provenientes de la red de sensores globales **WWLLN**, que detecta el origen de las descargas mediante técnicas combinadas de Tiempo de Arribo (TOA) y Tiempo de Arribo de Grupo (TOGA) en la banda de muy baja frecuencia (VLF) [11], [12].

El dataset está compuesto por archivos planos estructurados (`.loc`), cuyas características principales extraídas por cada evento son:
* **Timestamp:** Estampa de tiempo de alta precisión (resolución en microsegundos, UTC).
* **Coordenadas Geográficas:** Latitud y longitud calculadas en grados decimales.
* **Residual:** Error de ajuste temporal del algoritmo de localización de la red (restringido a menos de 30 microsegundos).
* **Estaciones (Nstn):** Cantidad de estaciones receptoras de la red que validaron el impacto (mayor o igual a 5).

### Pipeline de Ingeniería de Datos Planificado:
Debido a que el volumen global saturaría el entorno de ejecución, se ha diseñado un pipeline riguroso en Python para su posterior implementación:
1. **Delimitación por Bounding Box:** Se realizará un recorte espacial para aislar únicamente el territorio colombiano (Latitud: -4° a 13° N; Longitud: -79° a -67° W).
2. **Discretización Temporal (Time-Binning):** Los impactos de rayos se agruparán cronológicamente en ventanas discretas fijas de **10 minutos**.
3. **Rasterización a Matrices Numéricas:** Cada ventana se transformará en una cuadrícula (grid) bidimensional de dimensiones H x W (donde cada pixel equivale a una resolución de 0.1° x 0.1° geográficos). Si ocurre una o más descargas en esa coordenada dentro del bloque de tiempo, el pixel tomará el valor de **1**; en caso contrario, tomará el valor de **0**.

---

## 4. Métodos
El fenómeno se formula matemáticamente como un problema de predicción de secuencia de video continuo (*Spatiotemporal Sequence Prediction*).

* **Entrada de la Red (X):** Un tensor de dimensión (Batch, T, Canales, Alto, Ancho). Representará una secuencia consecutiva de **6 mapas binarios** correspondientes a los últimos 60 minutos de historial de la tormenta (donde T=6, Canales=1, y H x W estará adaptado a la resolución espacial).
* **Salida de la Red (Y):** Un tensor de dimensión (Batch, 1, Alto, Ancho), que predecirá el mapa de probabilidad de impactos para el bloque temporal inmediatamente posterior (T + 10 minutos).

### Arquitectura propuesta:
Se implementará una arquitectura basada en capas **ConvLSTM2D** empiladas, seguidas por una capa convolucional final de salida con activación *Sigmoid*. La ventaja crítica de este diseño radica en que las compuertas de la memoria LSTM clásica se sustituyen por operaciones de convolución local. Esto permitirá que el modelo mantenga un estado de memoria a largo plazo (capturando la inercia temporal del sistema meteorológico) sin perder las relaciones de adyacencia espacial (esenciales para modelar la propagación de los frentes de ráfaga hacia celdas vecinas).

### Métrica de Evaluación del Modelo:
Dada la naturaleza dispersa de los rayos, el dataset sufre de un desbalance de clases extremo (la gran mayoría del territorio nacional registrará un valor de cero en un minuto dado). Un clasificador ingenuo que prediga un mapa vacío obtendría un acierto analítico (*Accuracy*) artificial superior al 95%. Para solucionar este sesgo, se definen metodológicamente:
* **Función de Pérdida (Loss):** *Weighted Binary Cross-Entropy* (Entropía Cruzada Binaria Ponderada), escalando la penalización de los errores sobre los pixeles activados (1) para forzar a la red a enfocarse en los núcleos de tormenta.
* **Métricas de Desempeño:** *Critical Success Index* (CSI o *Threat Score*) y el *F1-Score / Coeficiente Dice*, los cuales omiten los verdaderos negativos del fondo del mapa y evalúan exclusivamente la precisión y sensibilidad de los frentes de descarga predichos.

---
## 5. Resultados (Avances de la Fase de Ingeniería de Datos)
A diferencia de los enfoques teóricos puros, esta primera entrega cuenta con un pipeline funcional de ingeniería de datos en Python compuesto por tres scripts operacionales, cuyos resultados de diseño e implementación son:

* **Optimización de Memoria mediante Chunking:** Con el script `prefilter_Colombia.py` se implementó una estrategia de lectura por bloques de 500,000 filas. Esto permite procesar archivos masivos de la red WWLLN (como el archivo de prueba `A20260121.loc`) aislando instantáneamente los eventos dentro de la caja de delimitación de Colombia (Longitud: -79.1° a -66.85°; Latitud: -4.5° a 13.6°) sin saturar la memoria RAM.
* **Filtrado Geospacial Fino con Buffer de Seguridad:** Mediante el script `filter_Colombia.py` y el uso de las librerías `Geopandas` y `Shapely`, se logró cargar la capa oficial de departamentos (`Departamentos_Septiembre_2025.gpkg`), unificarla (*dissolve*) y proyectarla al sistema EPSG:3116 (MAGNA-SIRGAS / Bogotá). Se integró con éxito un buffer perimetral parametrizable de 20 km para no descartar descargas en zonas costeras o fronterizas, reduciendo eficientemente los puntos crudos a un subset estrictamente nacional (`Colombia_A20260121.loc`).
* **Garantía de Secuencialidad Cronológica:** El script `time_sorting.py` resolvió mediante expresiones regulares (`re`) el ordenamiento estable de las estampas de tiempo de los rayos con precisión de microsegundos (`HH:MM:SS.ffffff`). Esto asegura que las ráfagas ingresen a la red neuronal respetando la causalidad del tiempo, aislando además registros corruptos al final del archivo.

---

## 6. Conclusiones y Trabajo Futuro
1. La arquitectura de datos diseñada demuestra que el manejo de Big Data meteorológico se vuelve viable al implementar pre-filtros por caja de delimitación (*Bounding Box*) antes de realizar operaciones de intersección de polígonos complejas, reduciendo drásticamente el costo computacional.
2. El uso de buffers espaciales proyectados (20 km en EPSG:3116) es metodológicamente indispensable en el estudio de descargas atmosféricas para capturar la dinámica de frentes de tormenta que se aproximan desde las fronteras terrestres o marítimas de Colombia.
3. Con el pipeline de limpieza, filtrado y ordenamiento cronológico completamente terminado y operativo, el trabajo futuro inmediato se enfocará en la fase de **Rasterización** (transformar estos archivos `.loc` filtrados en tensores matriciales 5D de ceros y unos) para iniciar formalmente el entrenamiento de la red ConvLSTM2D utilizando aceleración por GPU en el entorno de Google Colab.
---

## 7. Enlace al Repositorio de GitHub
* **Repositorio del Proyecto:** [Inserta aquí la URL de tu repositorio de GitHub para el Prof. Arles]

---

## Referencias
* **[1]** V. Cooray, *An Introduction to Lightning*. Springer, 2015.
* **[2]** V. Rakov and M. a. Uman, *Lightning: Physics and Effects*. Cambridge University Press, 2007.
* **[3]** T. J. Lang et al., "WMO world record lightning extremes," *BAMS*, 2017.
* **[4]** R. Albrecht et al., "Where are the lightning hotspots on Earth?," *BAMS*, 2016.
* **[5]** D. J. Cecil et al., "TRMM LIS climatology of thunderstorm occurrence," *Journal of Climate*, 2015.
* **[6]** A. S. Cruz-Bernal et al., "Lightning mortality rate in Colombia for the period 1997 – 2014," *Revista UIS Ingenierías*, 2018.
* **[7]** A. S. Cruz-Bernal, "Evaluación del riesgo por rayos para Colombia," *Universidad Nacional de Colombia*, 2019.
* **[8]** D. Aranguren et al., "Natural observatories for lightning research in Colombia," *ICEAA*, 2018.
* **[9]** J. O. Kaplan and K. H.-K. Lau, "WWLLN Global Lightning Climatology (WGLC) and time series, 2022 update," *Earth System Science Data*, 2022.
* **[10]** Díaz-Ortiz, Romero-Romero, Yepes-Chala y Cifuentes-Guerrero, "Web application for lightning occurrence visualization: Lightspot," *Ingeniería y Competitividad*, vol. 27, no. 1, 2025.
* **[11]** A. Alammari et al., "Lightning Mapping: Techniques, Challenges, and Opportunities," *IEEE Access*, 2020.
* **[12]** F. Lyu et al., "A low-frequency near-field interferometric-TOA 3-D Lightning Mapping Array," *Geophysical Research Letters*, 2014.